# judge

> Claude's labels of which sentences have a subject that does not do what its verb says, and how well models agree with them

Some sentences are hard to follow because their subject does not do what the verb says. "Rows carry their source" is one. Rows never normally carry anything. The writer means that each row has a field naming where it came from.

The rules in `syntax` find some of these sentences from lists of verbs and nouns. This notebook holds sentences that Claude labelled by reading each one, with no rule to decide the label. We use them to test whether a model can make the same judgement.

## Example pairs

Each pair gives a sentence that is hard to follow and a plain version of the same fact. Claude wrote the pairs for use as prompt examples. Jeremy has reviewed them. None comes from a document that we test on.

In [ ]:
pairs = [
    ('Headers ride along with each request.', 'Each request includes its headers.'),
    ('Startup latency demands a warm process.', 'We keep one process warm so that the command starts quickly.'),
    ('0.8 keeps roughly a third of the candidates.', 'With a threshold of 0.8, the filter keeps about a third of the candidates.'),
    ('The quiet path skips validation.', 'When `quiet` is set, `load` skips validation.'),
    ("The positions decide the result's shape.", 'Pass `np.take` an array of indices. It returns an array with the same shape as the indices.'),
    ('Trust lives in the signature check.', 'We trust a file only after `verify` checks its signature.'),]

These sentences are fine as they stand. In order, their subjects are a program that performs the action, a measurement and a person:

In [ ]:
fine = [
    'The parser rejects malformed input.',
    'The error rate is about two percent.',
    'I read every match by hand and removed the false ones.',]

## Labelled sentences

`samples/judge.jsonl` holds every prose sentence of four notebooks as they stood when Claude labelled them. `group` is `A` when the subject of a clause does not do what its verb says, as in the group A examples of `slopometer/judge_groups.txt`. It is `B` when the sentence has no such clause, whatever else is wrong with it. It is null when Jeremy placed the sentence in between. Claude chose the groups by reading. `jeremy` holds Jeremy's own words for each sentence he has judged, and the other rows have had no review.

The `dev` rows come from `08_corpus.ipynb`. We tune prompts on them. The `holdout` rows come from `00_core.ipynb`, `01_segment.ipynb` and `06_cli.ipynb`. We check a finished prompt on them once, so that they show how it does on sentences it was not tuned for.

In [ ]:
from fastcore.utils import *
from nbdev.config import get_config
import json
from collections import Counter

In [ ]:
rows = L((get_config().config_path/'samples'/'judge.jsonl').read_text().splitlines()).map(json.loads)
Counter((r['split'], r['group']) for r in rows)

A few of the group A sentences in the holdout rows:

In [ ]:
rows.filter(lambda r: r['group']=='A' and r['split']=='holdout').attrgot('text')[:5]

## Model survey

`judge_prompt` shows a model the example pairs and the fine sentences, then asks for `H` or `F` on every numbered sentence, the verdict form that `slopometer.ai.parse_verdicts` reads. The prompt has no description of the fault beyond its first sentence.

In [ ]:
import asyncio, time
from fastllm.acomplete import acomplete
from fastllm.chat import contents
from aidialog.msg_parts import mk_msgs
from fastcore.meta import delegates
from slopometer.ai import parse_verdicts

In [ ]:
def judge_prompt(sents, pairs=pairs, fine=fine):
    "Prompt asking for an `H` or `F` verdict on each of `sents`"
    ex = '\n\n'.join(f'Hard: {h}\nPlain: {p}' for h,p in pairs)
    ok = '\n'.join(f'Fine: {s}' for s in fine)
    doc = '\n'.join(f'[{i}] {s}' for i,s in enumerate(sents))
    return f"""Some sentences make the reader stop and work out what really happened and who did it. Each example below gives a hard sentence and a plain version of the same fact.

{ex}

These sentences are fine as they stand.

{ok}

Judge each numbered sentence below in the same way. Answer with one line per sentence, in the form `12 H` for a hard sentence or `12 F` for a fine one, and nothing else.

{doc}"""

In [ ]:
print(judge_prompt(['The parser reads the file.', 'Offsets ride in each row.']))

In [ ]:
from fastcore.test import *

`survey` runs `judge` several times for each model, all in parallel, and keeps an error as a string so that one failing model does not stop the rest. `tally` counts the hard sentences a model found, the fine sentences it wrongly called hard, and the sentences it gave no verdict for. Model names are `fastllm` vendor paths such as `openai/gpt-6-luna`.

In [ ]:
KEYS = dict(anthropic=os.environ.get('ANTHROPIC_KEY'))
@delegates(acomplete)
async def judge(model, sents, timeout=450, **kwargs):
    "`model`'s verdicts on `sents`, with the seconds taken"
    t = time.time()
    c = await asyncio.wait_for(acomplete(mk_msgs([judge_prompt(sents)]), model, api_key=KEYS.get(model.split('/')[0]), **kwargs), timeout)
    return parse_verdicts(contents(c).text), round(time.time()-t, 1)

def tally(verdicts, labelled):
    "Group A sentences found, group B sentences wrongly flagged, and sentences with no verdict; rows with no group count in neither"
    a,b = [{i for i,r in enumerate(labelled) if r['group']==g} for g in 'AB']
    said = {i for i,v in verdicts.items() if v}
    return dict(found=len(said&a), wrong=len(said&b), missing=len(labelled)-len(verdicts))

@delegates(judge)
async def survey(models, labelled, n=3, **kwargs):
    "Run `judge` `n` times per model in parallel; one result list per model, with errors kept as strings"
    sents = [r['text'] for r in labelled]
    async def one(m):
        try:
            v,s = await judge(m, sents, **kwargs)
            return dict(tally(v, labelled), secs=s, verdicts=v)
        except Exception as e: return dict(error=f'{type(e).__name__}: {str(e)[:150]}')
    res = await asyncio.gather(*[one(m) for m in models for _ in range(n)])
    return {m: res[i*n:(i+1)*n] for i,m in enumerate(models)}

The models are the solve-lp catalog, with each model once. The cell calls paid APIs and takes several minutes, so the docs build does not run it:

In [ ]:
#| eval: false
from solve_lp.ai import model_specs, provider_model
models = sorted({provider_model(m) for m in model_specs})
dev = rows.filter(lambda r: r['split']=='dev')
res = await survey(models, dev)
{m: [(r['found'], r['wrong']) if 'error' not in r else r['error'][:30] for r in rs] for m,rs in res.items()}

Run on 11 November 2026 against the 51 `dev` sentences, the survey gave these ranges over three runs. "Found" is out of 18 hard sentences and "wrong" is out of 33 fine ones. The labels are Claude's alone, so the table measures agreement with one reader.

| Model | Found | Wrong | Seconds per call |
|---|---|---|---|
| `gpt-6-astra` | 16 to 18 | 7 to 8 | 35 to 51 |
| `gemini-3.8-flash` | 14 to 15 | 2 to 4 | 155 to 231 |
| `gemini-4-argon` | 11 to 14 | 0 to 2 | 7 to 10 |
| `claude-fable-5-1` | 12 to 17 | 3 to 5 | 29 to 50 |
| `gpt-6.1-sol` | 11 to 16 | 3 to 6 | 32 to 42 |
| `claude-opus-5-5` | 11 to 16 | 3 to 4 | 11 to 13 |
| `gpt-6.1-terra` | 8 to 17 | 1 to 6 | 20 to 27 |
| `qwen3p8-max` | 7 to 14 | 3 to 6 | 129 to 156 |
| `claude-sonnet-5-5` | 7 | 0 to 2 | 12 to 13 |
| `gemini-3.5-flash-lite` | 7 to 14 | 1 to 11 | 2 to 3 |
| `gpt-6-luna` | 9 to 10 | 10 to 14 | 22 to 29 |
| `deepseek-v4p1-flash` | 3 to 10 | 1 to 2 | 99 to 117 |
| `glm-5p3-flash` | 0 to 5 | 2 to 5 | 117 to 296 |
| `claude-haiku-5-5` | 1 to 3 | 2 | 8 to 9 |
| `minimax-m3p1-flash` | 0 to 2 | 0 to 1 | 67 to 87 |

`muse-spark-1.3` and `muse-spark-1.3-contributor` each answered once and timed out twice. `glm-5p3` and `kimi-k3` timed out in every run. Fireworks returned a 404 for `qwen3p7-plus`.

`survey` passes extra options to `acomplete`. On the same day, `survey(['gemini/models/gemini-3.8-flash'], dev, reasoning_effort=...)` gave these results for each run. The model's metadata lists three levels, so the fourth row is the default with no setting.

| Level | Found | Wrong | Seconds |
|---|---|---|---|
| default | 10, 11, 10 | 2, 8, 2 | 211, 9, 169 |
| `low` | 15, 15, 16 | 5, 3, 8 | 6, 9, 8 |
| `medium` | 11, 11, 12 | 2, 2, 2 | 122, 244, 105 |
| `high` | 15 | 6 | 293 |

Two of the three `high` runs timed out at 450 seconds. The default found 10 or 11 here and 14 or 15 in the survey above, with the same prompt. Three runs are too few to rank the levels.

## Jev

TypeSafe's Jev model answers a yes/no question with a probability and generates no text. This section asks it whether a sentence fits one of two example groups. `slopometer/judge_groups.txt` holds the groups. Group A holds sentences of the hard kind. Group B holds plain sentences, some of them unrelated to the rest. The question names the groups only as `grammar_group_a` and `grammar_group_b`, so that it describes no fault.

In [ ]:
from slopometer.typesafe import *
from slopometer.segment import scrub
from slopometer.syntax import get_nlp

In [ ]:
from slopometer.ai import groups
ga,gb = groups()
ga[:3], gb[:3]

A long sentence can hold several clauses, and only one of them may be hard to follow. Every group A example is a single short clause. `clauses` splits a sentence into its subject-and-verb clauses, so that Jev can judge each one against examples of the same size:

In [ ]:
_cut = {'conj', 'advcl', 'parataxis', 'cc', 'mark', 'punct', 'appos', 'relcl', 'acl'}
def clauses(txt):
    "Each subject-and-verb clause of `txt`, without the clauses joined to it; `[txt]` when it has none"
    d,res = get_nlp()(scrub(txt)),[]
    def walk(t, keep, top=False):
        keep.add(t.i)
        for c in t.children:
            if not (top and c.dep_ in _cut): walk(c, keep)
    for v in d:
        subs = [c for c in v.children if c.dep_ in ('nsubj', 'nsubjpass')]
        if not subs or v.pos_ not in ('VERB', 'AUX'): continue
        keep = set()
        walk(v, keep, True)
        for s in subs: walk(s, keep)
        ids,out = sorted(keep),''
        for a,b in zip(ids, ids[1:]+[None]):
            out += txt[d[a].idx:d[a].idx+len(d[a].text)]
            if b is not None: out += d[a].whitespace_ if b == a+1 else ' '
        res.append(out.strip())
    return res or [txt]

In [ ]:
cs = clauses('Rows carry their source, so any experiment can slice by it later.')
test_eq(cs[0], 'Rows carry their source')
cs

Each question builder returns the Noul questions for one labelled row. `sent_q` asks about the sentence alone. `nbr_q` adds the sentences before and after it as separate named fields. `clause_q` asks one question per clause and shows the full sentence beside it. In each, the field name in the question says which text Jev must judge.

In [ ]:
JQ = 'Does `{}` fit in `grammar_group_a`, grammatically?'
def sent_q(r, drows): return [noul(dict(sentence=r['text'], question=JQ.format('sentence')))]
def nbr_q(r, drows):
    nb = lambda k: drows[r['i']+k]['text'] if 0 <= r['i']+k < len(drows) else None
    ins = dict(sentence_before=nb(-1), sentence_to_judge=r['text'], sentence_after=nb(1), question=JQ.format('sentence_to_judge'))
    return [noul({k: v for k,v in ins.items() if v is not None})]
def clause_q(r, drows):
    return [noul(dict(full_sentence=r['text'], clause_to_judge=c, question=JQ.format('clause_to_judge'))) for c in clauses(r['text'])]

In [ ]:
row = dict(i=0, text='Rows carry their source, so any experiment can slice by it later.')
test_eq(len(clause_q(row, [row])), 2)
clause_q(row, [row])[0]

`jev_probs` sends one request per document, with the two groups as the shared state. A row's probability is the highest that any of its questions received. `rank_score` is the chance that a hard row gets a higher probability than a fine one. It is 0.5 for a random scorer and 1.0 for a perfect one. `flagged` counts the hard rows found and the fine rows wrongly flagged at a threshold.

In [ ]:
def jev_probs(labelled, mkqs, state):
    "Jev's probability that each row fits group A: the highest over the questions `mkqs(row, doc_rows)` builds"
    res = {}
    for d in dict.fromkeys(r['doc'] for r in labelled):
        drows = [r for r in labelled if r['doc']==d]
        ans = ask_nouls(state, {f"s{r['i']}_{k}": q for r in drows for k,q in enumerate(mkqs(r, drows))})
        for r in drows: res[d, r['i']] = max(v for k,v in ans.items() if k.startswith(f"s{r['i']}_"))
    return [res[r['doc'], r['i']] for r in labelled]

def rank_score(ps, labelled):
    "Chance that a group A row gets a higher probability than a group B row, counting a tie as half"
    h,f = [[p for p,r in zip(ps, labelled) if r['group']==x] for x in 'AB']
    return round(sum((x>y) + 0.5*(x==y) for x in h for y in f)/(len(h)*len(f)), 3)

def flagged(ps, labelled, thr=0.7):
    "Group A rows found and group B rows wrongly flagged above `thr`"
    return tuple(sum(p>thr and r['group']==x for p,r in zip(ps, labelled)) for x in 'AB')

In [ ]:
lab = [dict(group='A'), dict(group='B'), dict(group='A'), dict(group=None)]
test_eq(rank_score([0.9, 0.2, 0.2, 0.9], lab), 0.75)
test_eq(flagged([0.9, 0.8, 0.2, 0.9], lab), (1, 1))

The cell calls a paid API, so the docs build does not run it:

In [ ]:
#| eval: false
state = dict(grammar_group_a=ga, grammar_group_b=gb)
dev,holdout = (rows.filter(lambda r: r['split']==s) for s in ('dev', 'holdout'))
{f.__name__: (rank_score(ps, dev), flagged(ps, dev)) for f in (sent_q, nbr_q, clause_q) for ps in [jev_probs(dev, f, state)]}